# Fine-tune GPT-2 to Generate Quotes

This notebook fine-tunes the small GPT-2 model on a list of short inspirational quotes, then uses it to generate new quotes from a starting word. It uses a simple PyTorch training loop so every step is visible. The dataset is tiny, so this is a learning exercise, not a production model.

## How to run
- Open in Google Colab. Runtime type: **T4 GPU (CPU also works, just slower)**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `Untitled5.ipynb` and `Untitled6.ipynb`._

In [ ]:
!pip install -q transformers torch

## 1. Training data
A small list of quotes. Add your own lines to make the model better.

In [ ]:
quotes = [
    "The only way to do great work is to love what you do.",
    "Believe you can and you're halfway there.",
    "Success is not final, failure is not fatal: it is the courage to continue that counts.",
    "Love is the bridge between two hearts.",
    "Happiness depends upon ourselves.",
    "Dream big and dare to fail.",
    "Act as if what you do makes a difference. It does.",
    "Keep your face always toward the sunshine and shadows will fall behind you.",
    "It always seems impossible until it is done.",
    "Love all, trust a few, do wrong to none.",
    "The best time to plant a tree was 20 years ago. The second best time is now.",
    "Life is what happens when you are busy making other plans.",
]
print(len(quotes), "quotes")

## 2. Load GPT-2 and build a dataset
GPT-2 has no padding token, so we reuse the end-of-text token. Padding positions get label `-100` so the loss ignores them.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

class QuotesDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=48):
        self.enc = tokenizer(
            [t + tokenizer.eos_token for t in texts],
            max_length=max_length, padding="max_length", truncation=True, return_tensors="pt",
        )

    def __len__(self):
        return self.enc["input_ids"].shape[0]

    def __getitem__(self, i):
        input_ids = self.enc["input_ids"][i]
        attention_mask = self.enc["attention_mask"][i]
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

loader = DataLoader(QuotesDataset(quotes, tokenizer), batch_size=4, shuffle=True)

## 3. Train
A plain loop with `torch.optim.AdamW` and a linear learning-rate schedule.

In [ ]:
from transformers import get_linear_schedule_with_warmup

epochs = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0,
                                            num_training_steps=epochs * len(loader))

model.train()
for epoch in range(epochs):
    total = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total += loss.item()
    print(f"Epoch {epoch + 1}/{epochs} - loss {total / len(loader):.3f}")

## 4. Save the model

In [ ]:
model.save_pretrained("fine_tuned_gpt2_quotes")
tokenizer.save_pretrained("fine_tuned_gpt2_quotes")

## 5. Generate quotes
Sampling (`do_sample=True`) with `top_p` gives different quotes each time.

In [ ]:
model.eval()

def generate_quotes(prompt, num_quotes=3, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True, top_p=0.92, temperature=0.8,
            num_return_sequences=num_quotes,
            pad_token_id=tokenizer.eos_token_id,
        )
    return [tokenizer.decode(o, skip_special_tokens=True).strip() for o in outputs]

for q in generate_quotes("Love"):
    print("-", q)

## 6. Optional: a small Gradio app

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def ui(prompt, n):
    return "\n\n".join(generate_quotes(prompt, num_quotes=int(n)))

gr.Interface(fn=ui,
             inputs=[gr.Textbox(label="Starting word", value="Life"),
                     gr.Slider(1, 5, value=3, step=1, label="How many")],
             outputs=gr.Textbox(label="Generated quotes"),
             title="GPT-2 Quote Generator").launch()